In [1]:
%load_ext autoreload
%autoreload 2
import dask
import dask.distributed
from dask_util import DaskClient
import dask_util
import numpy as np

In [2]:
local_params = {
    "n_workers":4, 
    "processes" : True, 
    "dashboard_address" : 'localhost:7777'
    
}

# cluster = {
#     "cores" : 24,
#     "processes" : 1,
#     "memory" : "1GB",
#     "shebang" : '#!/usr/bin/env bash',
#     "queue" : "serc",
#     "walltime" : "00:10:00",
#     "local_directory" : '/tmp',
#     "death_timeout" : "15s",
#     "interface" : "ib0",
#     "log_directory" : f'{os.environ["SCRATCH"]}/dask_jobqueue_logs/'    
# }


client = DaskClient(local_params=local_params)

In [3]:
%load_ext autoreload
%autoreload 2
import SepVector
from __pyDaskVector import DaskVector
from __pyDaskOperator import DaskOperator
import Hypercube
import pyOperator as Op


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
WARNING! DATAPATH not found. The folder /tmp will be used to write binary files


/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:500: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.9/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


In [4]:

ns = [10,4]
os = [0,0]
ds = [1,1]
chunks = (1,3)

ax = Hypercube.axis(n=1, o=1, d=1)
hyp = Hypercube.hypercube(ns=ns, ds=ds, os=os)
vec = SepVector.getSepVector(ns=ns, ds=ds, os=os)
vec.set(1)

floatVector
Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=4	o=0.000000	d=1.000000

In [5]:
# option 1
# creating from scratch
data = DaskVector(client, vecCls=SepVector.floatVector, ns=ns, ds=ds, os=os, chunks=chunks)

In [6]:
data[:]

[array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)]

In [7]:
# option 2
# creating from existing in-memory SepVector
daskVec = DaskVector(client, from_vector=vec, chunks=chunks)

In [8]:
client.getClient().has_what()

{'tcp://127.0.0.1:34481': ('floatVector-d59684f11f14cb3caa5c1d479b84226a',
  'window-35893685a4e24fb3d67cb08e004144f0'),
 'tcp://127.0.0.1:35189': ('floatVector-c2854c5283feeff2e1cb2c07fd755314',
  'window-f9605a7b6fba6c09467ec35abcf7b8cb'),
 'tcp://127.0.0.1:38511': ('floatVector-43c6fc95e2e590b9bf58e2f816195d75',),
 'tcp://127.0.0.1:39165': ('window-b624802f8e5361dfbe2cf4d213f34dac',)}

In [9]:
daskVec.getHyper()

Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=4	o=0.000000	d=1.000000

In [10]:
scaleOp = DaskOperator(client, Op.ScalingOp, daskVec, data, 4)

AttributeError: module 'pyOperator' has no attribute 'ScalingOp'

In [ ]:
data[:]
daskVec.set(1)

In [ ]:
scaleOp.forward(False, daskVec, data)

In [ ]:
data[:]

In [ ]:
scaleOp.adjoint(False, daskVec, data)

In [ ]:
daskVec[:]